# Diabetes Prediction — Exploratory Data Analysis

This notebook explores the Pima Indians Diabetes dataset before model training.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.dataset import FEATURE_NAMES, ZERO_AS_MISSING, load_raw

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
df_raw = pd.read_csv('../data/raw/diabetes.csv')
print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
df_raw.describe().T

## 2. Class Distribution

In [ ]:
counts = df_raw['Outcome'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['No Diabetes (0)', 'Diabetes (1)'], counts.values,
              color=['steelblue', 'tomato'], edgecolor='black')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(int(bar.get_height())), ha='center', fontsize=11)
ax.set_ylabel('Count')
ax.set_title('Class Distribution')
plt.tight_layout()
plt.savefig('../outputs/figures/class_distribution.png', dpi=150)
plt.show()

## 3. Missing Values (biologically impossible zeros)

In [ ]:
df = load_raw('../data/raw/diabetes.csv')
missing = (df.isnull().sum() / len(df) * 100).round(1)
print('Missing % after zero-replacement:')
print(missing[missing > 0])

## 4. Feature Distributions by Outcome

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, feat in zip(axes.flatten(), FEATURE_NAMES):
    for outcome, color, label in [(0, 'steelblue', 'No Diabetes'), (1, 'tomato', 'Diabetes')]:
        subset = df[df['Outcome'] == outcome][feat].dropna()
        subset.plot(kind='kde', ax=ax, color=color, label=label, linewidth=2)
    ax.set_title(feat)
    ax.legend(fontsize=7)
plt.suptitle('Feature Distributions by Outcome', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/feature_distributions.png', dpi=150)
plt.show()

## 5. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df_raw.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150)
plt.show()

## 6. Boxplots — Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, feat in zip(axes.flatten(), FEATURE_NAMES):
    df_raw.boxplot(column=feat, by='Outcome', ax=ax)
    ax.set_title(feat)
    ax.set_xlabel('Outcome')
plt.suptitle('Boxplots by Outcome', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/boxplots.png', dpi=150)
plt.show()

## 7. Key Observations

- **Class imbalance**: ~65% No Diabetes, ~35% Diabetes → handled with SMOTE in training.
- **Missing data**: Glucose, BloodPressure, SkinThickness, Insulin, BMI contain biological zeros → replaced with NaN and imputed (median).
- **Most separating features**: Glucose, BMI, and Age show the clearest distributional differences between classes.
- **Outliers**: Insulin and SkinThickness have heavy tails; tree-based models handle this robustly.
